# TODO 6.3：GPTQ 质量与 AWQ 性能对比

对 GPTQ g32/g64/g128 执行完整 FinQA 质量和高风险回归，并在同一张 GPU 上对 GPTQ/AWQ 执行 batch 1/4/8/16 的统一 Benchmark。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import gc, json, sys
from pathlib import Path
import torch
PROJECT_DIR=Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR=Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST=Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
BASELINE_CONFIG=PROJECT_DIR/'configs/baseline.json'
QUALITY_CONFIG=PROJECT_DIR/'configs/baseline_quality.json'
BENCHMARK_CONFIG=PROJECT_DIR/'configs/quantization_benchmark.json'
GPTQ_DIR=PROJECT_DIR/'out/gptq'
AWQ_DIR=PROJECT_DIR/'out/awq'
sys.path.insert(0,str(PROJECT_DIR))
from src.baseline.service import BaselineService, InferenceRequest
from src.data.finqa_assets import load_assets
from src.evaluation.baseline_quality import run_baseline_quality, validate_quality_report
from src.evaluation.finqa_metrics import classify_numeric_regression
from src.benchmark.runner import run_benchmark, validate_benchmark_report
from src.quantization.gptq import load_gptq_service
from src.quantization.awq import load_awq_service
assets=load_assets(DATASET_DIR)
quality_config=json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))
benchmark_config=json.loads(BENCHMARK_CONFIG.read_text(encoding='utf-8'))
print(torch.cuda.get_device_name(0),len(assets.quality_dev),len(assets.high_risk_regression))

In [ ]:
def predict_rows(service,dataset,prefix):
    rows=[]; batch_size=int(quality_config['batch_size'])
    for start in range(0,len(dataset),batch_size):
        batch=[dataset[index] for index in range(start,min(start+batch_size,len(dataset)))]
        requests=[InferenceRequest.from_finqa_row(row,request_id=f'{prefix}-{start+offset+1:06d}') for offset,row in enumerate(batch)]
        results=service.generate_batch(requests,prompt_suffix=quality_config.get('prompt_suffix',''))
        for row,request,result in zip(batch,requests,results):
            rows.append({'question':request.question,'prediction':result['output_text'],'reference':row.get('answer') or row.get('qa',{}).get('answer')})
    return rows
baseline=BaselineService.from_local_model(BASELINE_CONFIG,PROJECT_DIR/'out/model_input_real/model_manifest.json',GPTQ_DIR/'fp16_regression_service.jsonl')
baseline_regression=predict_rows(baseline,assets.high_risk_regression,'fp16-regression')
del baseline; gc.collect(); torch.cuda.empty_cache()

In [ ]:
for group_size in (32,64,128):
    plan_id=f'gptq-int4-w4a16-g{group_size}'; plan_dir=GPTQ_DIR/plan_id
    service=load_gptq_service(BASELINE_CONFIG,plan_dir/'quantized_model_manifest.json',MODEL_MANIFEST,plan_dir/'gptq_quality_service.jsonl')
    quality_dir=plan_dir/'quality'
    quality=run_baseline_quality(service,assets.quality_dev,assets.manifest,quality_config,quality_dir)
    quality_file=quality_dir/'baseline_quality_report.json'
    quality_file.write_text(json.dumps(quality,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    validation=validate_quality_report(quality_file)
    (quality_dir/'validation.json').write_text(json.dumps(validation.to_dict(),ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    candidate=predict_rows(service,assets.high_risk_regression,f'{plan_id}-regression')
    baseline_map={row['question']:row for row in baseline_regression}
    comparisons=[{**classify_numeric_regression(baseline_map[row['question']]['prediction'],row['prediction'],row['reference'],float(quality_config.get('tolerance',1e-4))),'question':row['question']} for row in candidate]
    regression={'schema_version':1,'plan_id':plan_id,'dataset_role':'high_risk_regression','total':len(comparisons),'new_regressions':sum(row['new_regression'] for row in comparisons),'critical_errors':sum(row['critical_error_type'] is not None for row in comparisons),'rows':comparisons}
    (plan_dir/'regression.json').write_text(json.dumps(regression,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    print(plan_id,quality['metrics'],regression['new_regressions'])
    del service; gc.collect(); torch.cuda.empty_cache()

In [ ]:
for method,root,loader in [('gptq',GPTQ_DIR,load_gptq_service),('awq',AWQ_DIR,load_awq_service)]:
    for group_size in (32,64,128):
        plan_id=f'{method}-int4-w4a16-g{group_size}'; plan_dir=root/plan_id
        service=loader(BASELINE_CONFIG,plan_dir/'quantized_model_manifest.json',MODEL_MANIFEST,plan_dir/f'{method}_comparison_benchmark_service.jsonl')
        report=run_benchmark(service,assets.calibration_prompts[0],benchmark_config)
        report_file=plan_dir/'comparison_benchmark_report.json'
        report_file.write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
        validation=validate_benchmark_report(report_file)
        (plan_dir/'comparison_benchmark_validation.json').write_text(json.dumps(validation.to_dict(),ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
        assert validation.complete
        print(plan_id,'comparison benchmark PASS')
        del service; gc.collect(); torch.cuda.empty_cache()

完成后拉回 `out/gptq`，并拉回 `out/awq/*/comparison_benchmark_*`。完整 GPTQ/AWQ 权重不要放进项目仓库。